# AIC 2026 - Caption Qwen3-VL-8B Có Ngữ Cảnh Shot Trước (L24 / L25 / L26)

Nâng cấp so với `tune_qwen3vl_8b_prompt.ipynb`:
1. **Ngữ cảnh shot trước:** caption của shot hợp lệ gần nhất trong cùng video được đưa vào prompt, cùng với title và ASR. Nếu shot liền trước bị bỏ (quảng cáo/intro), notebook nhảy qua để lấy shot hợp lệ gần nhất. Vì vậy các shot trong một video phải chạy tuần tự.
2. **Chỉ chạy L24, L25, L26.**
3. **Bỏ quảng cáo L25:** các đoạn ads trong `L25_ads.json` được đổi từ `frame_idx` sang giây theo fps riêng của từng video (có video 25 fps, có video 29.97 fps) rồi nhúng thẳng vào notebook. Shot nào có khoảng `[start, end]` trong tên file chồng lấn với một đoạn ads sẽ bị bỏ.
4. **Bỏ intro L26:** bỏ shot `0001`–`0003` của mọi video L26.
5. **Sửa timestamp:** truyền `video_metadata` để Qwen3-VL dùng đúng fps thật (trước đây nó ngầm giả định fps=24).

⚠️ **CHẾ ĐỘ THỬ NGHIỆM (`TEST_MODE = True`):** chỉ chạy `L26_V225`, 10 shot đầu. Shot 1–3 là intro nên bị bỏ, còn 7 shot được caption. Mỗi shot được chạy hai lần, có và không có ngữ cảnh, để so sánh.


--- 
## BƯỚC 1: Kiểm Tra Tài Nguyên GPU Trên Kaggle
*(Khuyến nghị: Chọn **GPU T4 x2 (Dual T4)** trong phần Settings để model 8B chạy 100% trên 32GB VRAM)*

In [10]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Số lượng GPU phát hiện: {device_count}")
for i in range(device_count):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} | VRAM: {round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)} GB")
if device_count >= 2:
    print("🔥 Tuyệt vời! Dual GPU phát hiện, model 8B sẽ được tự động chia đều để chạy mượt mà không nghẽn RAM!")
print("=" * 60)

Thu Sep 24 07:44:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   39C    P8             11W /  500W |   32000MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## BƯỚC 2: Cấu Hình, Luật Bỏ Shot (Ads L25, Intro L26) & Danh Sách Shot


In [11]:
import os
import re
import json

# ==============================================================================
# 1. ĐƯỜNG DẪN (vast.ai)
# ==============================================================================
VIDEO_DIR = "/workspace/data/cut_videos/cut_videos"      # cut_videos/<video_id>/<shot>.mp4
MEDIA_INFO_DIR = "/workspace/data/media-info/media-info"
ASR_DIR = "/workspace/data/asr/asr"
OUTPUT_DIR = "/workspace"

# ==============================================================================
# 2. PHẠM VI CHẠY & ÉP XUNG ĐA STREAM BATCHING
# ==============================================================================
TEST_MODE = False                 # True: chỉ TEST_VIDEOS và TEST_FIRST_N shot đầu. False: toàn bộ L24/L25/L26
TEST_VIDEOS = ["L26_V225"]
TEST_FIRST_N = 10                # tính trên TẤT CẢ shot của video (kể cả shot bị bỏ)
VIDEO_PREFIXES = ("L24", "L26")

BATCH_SIZE = 7                   # Batch Size = 2 gom 2 video chạy song song trên 2x 5060 Ti
LOG_FILE = os.path.join(OUTPUT_DIR, "caption_progress.log")
STATUS_FILE = os.path.join(OUTPUT_DIR, "caption_status.txt")

OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "qwen3vl_8b_ctx_test.jsonl" if TEST_MODE else "qwen3vl_8b_ctx_L24_L26.jsonl")

# ==============================================================================
# 3. LUẬT BỎ SHOT
# ==============================================================================
L26_INTRO_MAX_SHOT = 3           # L26_* luôn bỏ 3 shot đầu (shot 1, 2, 3)

# L25: Bảng thời gian (giây) của 3 cụm quảng cáo [start, end] cho từng video
L25_ADS_SEC = {
    "L25_V001": [[33.2, 93.16], [586.28, 669.08], [1160.72, 1219.64]],
    "L25_V002": [[24.52, 84.44], [640.88, 723.68], [1287.4, 1346.28]],
    "L25_V003": [[27.68, 87.6], [748.28, 831.08], [1467.56, 1526.48]],
    "L25_V004": [[34.92, 94.88], [447.88, 530.68], [911.36, 970.28]],
    "L25_V005": [[44.52, 104.44], [572.6, 655.4], [1096.72, 1155.64]],
    "L25_V006": [[54.68, 114.64], [520.48, 603.28], [1103.56, 1162.48]],
    "L25_V007": [[59.16, 119.08], [751.24, 834.04], [1328.64, 1387.56]],
    "L25_V008": [[36.72, 96.64], [708.28, 791.08], [1436.48, 1495.4]],
    "L25_V009": [[40.6, 100.52], [571.6, 654.4], [1085.6, 1144.52]],
    "L25_V010": [[28.52, 88.48], [480.96, 563.76], [1014.28, 1073.2]],
    "L25_V011": [[25.64, 85.6], [494.32, 577.12], [969.4, 1028.32]],
    "L25_V012": [[27.16, 87.08], [618.64, 701.44], [1182.2, 1241.12]],
    "L25_V013": [[33.23, 93.19], [423.29, 506.07], [945.31, 1004.14]],
    "L25_V014": [[35.67, 94.53], [586.25, 669.04], [1275.21, 1334.03]],
    "L25_V015": [[80.04, 139.96], [660.76, 743.56], [1265.08, 1324.08]],
    "L25_V016": [[23.72, 82.56], [699.0, 781.8], [1528.96, 1587.8]],
    "L25_V017": [[24.29, 84.25], [330.06, 412.85], [817.52, 876.34]],
    "L25_V018": [[47.6, 106.4], [894.16, 976.96], [1597.88, 1657.8]],
    "L25_V019": [[60.56, 119.4], [429.44, 512.24], [804.64, 864.56]],
    "L25_V020": [[220.76, 279.6], [622.16, 704.96], [1130.56, 1190.48]],
    "L25_V021": [[31.46, 90.29], [496.2, 578.98], [968.6, 1027.46]],
    "L25_V022": [[28.56, 87.39], [698.16, 780.95], [1352.32, 1412.28]],
    "L25_V023": [[181.96, 240.8], [872.6, 955.4], [1413.84, 1473.76]],
    "L25_V024": [[51.4, 110.24], [724.2, 807.0], [1370.0, 1429.92]],
    "L25_V025": [[24.06, 82.88], [588.72, 671.5], [1313.91, 1373.87]],
    "L25_V026": [[289.6, 349.52], [904.68, 987.48], [1586.76, 1645.6]],
    "L25_V027": [[76.08, 136.0], [938.48, 1021.28], [1715.72, 1774.56]],
    "L25_V028": [[70.8, 130.72], [377.56, 460.36], [786.76, 845.6]],
    "L25_V029": [[200.6, 260.52], [715.4, 798.2], [1291.44, 1350.28]],
    "L25_V030": [[29.86, 88.72], [586.32, 669.1], [1053.22, 1112.05]],
    "L25_V031": [[16.92, 76.88], [429.2, 511.98], [882.48, 941.31]],
    "L25_V032": [[214.0, 273.92], [955.32, 1038.12], [1597.72, 1656.56]],
    "L25_V033": [[29.4, 88.32], [930.08, 1012.88], [1864.72, 1924.64]],
    "L25_V034": [[27.19, 87.15], [491.56, 574.34], [1181.82, 1240.64]],
    "L25_V035": [[164.6, 223.44], [744.84, 827.64], [1235.24, 1295.16]],
    "L25_V036": [[59.32, 118.16], [947.24, 1030.04], [1741.92, 1801.84]],
    "L25_V037": [[47.56, 106.4], [383.4, 466.2], [756.24, 816.16]],
    "L25_V038": [[203.72, 262.56], [671.08, 753.88], [1301.32, 1361.24]],
    "L25_V039": [[31.46, 90.29], [650.72, 733.5], [1214.35, 1274.27]],
    "L25_V040": [[30.3, 89.12], [383.65, 464.9], [836.84, 896.8]],
    "L25_V041": [[283.0, 341.84], [789.12, 871.92], [1363.8, 1423.72]],
    "L25_V042": [[29.4, 88.32], [930.08, 1012.88], [1864.72, 1924.64]],
    "L25_V043": [[23.26, 82.08], [712.58, 795.36], [1352.42, 1412.38]],
    "L25_V044": [[101.92, 161.84], [612.48, 695.28], [1149.4, 1208.24]],
    "L25_V045": [[83.36, 143.28], [674.2, 757.0], [1317.84, 1376.68]],
    "L25_V046": [[70.2, 130.12], [388.28, 471.08], [821.28, 880.12]],
    "L25_V047": [[156.48, 216.4], [628.68, 711.48], [1262.44, 1321.28]],
    "L25_V048": [[25.16, 84.02], [854.72, 937.5], [1759.49, 1818.32]],
    "L25_V049": [[16.55, 75.41], [590.92, 673.71], [1318.05, 1376.88]],
    "L25_V050": [[198.56, 258.48], [823.24, 906.04], [1571.24, 1630.08]],
    "L25_V051": [[26.52, 86.44], [1070.64, 1153.44], [1953.84, 2012.68]],
    "L25_V052": [[37.44, 97.4], [554.75, 637.54], [1259.56, 1318.39]],
    "L25_V053": [[117.32, 176.16], [543.68, 626.48], [952.64, 1012.56]],
    "L25_V054": [[56.92, 115.76], [1179.6, 1262.4], [2453.68, 2513.6]],
    "L25_V055": [[43.96, 102.8], [493.96, 576.76], [927.0, 986.92]],
    "L25_V056": [[168.8, 227.64], [716.72, 799.52], [1366.6, 1426.52]],
    "L25_V057": [[35.14, 93.96], [454.15, 536.94], [934.17, 993.03]],
    "L25_V058": [[35.44, 94.26], [460.13, 542.91], [1096.03, 1154.72]],
    "L25_V059": [[235.96, 294.8], [704.6, 787.4], [1255.36, 1315.28]],
    "L25_V060": [[56.64, 115.44], [751.6, 834.4], [1518.28, 1578.2]],
    "L25_V061": [[26.69, 85.52], [423.82, 506.61], [1053.49, 1112.35]],
    "L25_V062": [[253.72, 313.64], [728.6, 811.4], [1188.04, 1246.88]],
    "L25_V063": [[54.44, 114.36], [671.76, 754.56], [1351.56, 1410.4]],
    "L25_V064": [[54.64, 114.56], [584.12, 666.92], [1103.32, 1162.16]],
    "L25_V065": [[105.44, 165.36], [532.32, 615.12], [1081.48, 1140.32]],
    "L25_V066": [[31.0, 90.96], [498.16, 580.95], [984.92, 1043.74]],
    "L25_V067": [[62.46, 122.42], [750.08, 832.33], [1386.95, 1445.78]],
    "L25_V068": [[241.16, 301.08], [907.44, 990.24], [1691.92, 1750.76]],
    "L25_V069": [[61.08, 121.0], [1030.92, 1113.72], [1875.08, 1933.92]],
    "L25_V070": [[33.43, 93.39], [438.57, 521.35], [1038.47, 1097.3]],
    "L25_V071": [[231.44, 290.28], [577.68, 660.48], [923.6, 983.52]],
    "L25_V072": [[67.92, 126.76], [437.36, 520.16], [827.28, 887.2]],
    "L25_V073": [[251.48, 310.32], [744.56, 827.36], [1271.44, 1331.36]],
    "L25_V074": [[37.67, 96.5], [848.75, 929.96], [1746.41, 1805.27]],
    "L25_V075": [[499.12, 557.96], [1080.0, 1162.8], [1671.64, 1731.56]],
    "L25_V076": [[65.28, 110.56], [833.72, 916.52], [1464.16, 1524.08]],
    "L25_V077": [[25.89, 84.72], [621.65, 704.44], [1374.31, 1434.27]],
    "L25_V078": [[38.17, 97.0], [665.87, 748.65], [1012.18, 1072.14]],
    "L25_V079": [[194.52, 254.44], [555.32, 638.12], [977.72, 1036.56]],
    "L25_V080": [[72.52, 131.36], [531.12, 613.92], [1353.84, 1413.76]],
    "L25_V081": [[174.56, 234.48], [1029.2, 1112.0], [2163.56, 2222.4]],
    "L25_V082": [[116.84, 176.76], [605.68, 688.48], [1057.88, 1116.72]],
    "L25_V083": [[249.84, 309.76], [647.64, 730.44], [1023.84, 1082.68]],
    "L25_V084": [[26.49, 86.45], [741.81, 824.59], [1876.08, 1934.9]],
    "L25_V085": [[59.39, 119.35], [920.52, 1003.3], [1929.96, 1988.79]],
    "L25_V086": [[545.52, 605.44], [1297.32, 1380.12], [2100.48, 2159.32]],
    "L25_V087": [[94.68, 154.6], [790.84, 873.64], [1566.04, 1624.88]],
    "L25_V088": [[31.33, 91.29], [496.93, 579.71], [1066.47, 1125.19]]
}

def parse_shot_filename(filename: str):
    """L26_V225_shot_0001_00m01s_00m03s.mp4 -> (video_id, shot_idx, start_sec, end_sec)."""
    stem = os.path.splitext(os.path.basename(filename))[0]
    parts = stem.split('_')
    if len(parts) < 6:
        return stem, 0, 0.0, 0.0
    def to_sec(ts):
        h = re.search(r'(\d+)h', ts); m = re.search(r'(\d+)m', ts); s = re.search(r'(\d+)s', ts)
        return float((int(h.group(1)) if h else 0) * 3600 + (int(m.group(1)) if m else 0) * 60 + (int(s.group(1)) if s else 0))
    return f"{parts[0]}_{parts[1]}", int(parts[3]), to_sec(parts[-2]), to_sec(parts[-1])

def skip_reason(filename: str):
    """Trả về lý do bỏ shot, hoặc None nếu shot được caption."""
    vid, idx, s, e = parse_shot_filename(filename)
    if vid.startswith("L26") and 1 <= idx <= L26_INTRO_MAX_SHOT:
        return "intro"
    for a, b in L25_ADS_SEC.get(vid, []):
        if s < b and e + 1 > a:    # chồng lấn; tên file làm tròn XUỐNG theo giây nên shot thật nằm trong [s, e+1)
            return f"ads[{a}-{b}s]"
    return None

# ==============================================================================
# 4. DANH SÁCH SHOT THEO TỪNG VIDEO (giữ đúng thứ tự shot để nối ngữ cảnh)
# ==============================================================================
if TEST_MODE:
    video_ids = TEST_VIDEOS
else:
    video_ids = sorted(d for d in os.listdir(VIDEO_DIR) if d.startswith(VIDEO_PREFIXES))

video_shots = {}                 # video_id -> [(path, skip_reason|None), ...] theo shot_idx
n_keep = n_intro = n_ads = 0
for vid in video_ids:
    vdir = os.path.join(VIDEO_DIR, vid)
    if not os.path.isdir(vdir):
        continue
    files = sorted((f for f in os.listdir(vdir) if f.endswith(".mp4")), key=lambda f: parse_shot_filename(f)[1])
    if TEST_MODE:
        files = files[:TEST_FIRST_N]
    lst = []
    for f in files:
        r = skip_reason(f)
        lst.append((os.path.join(vdir, f), r))
        if r is None: n_keep += 1
        elif r == "intro": n_intro += 1
        else: n_ads += 1
    video_shots[vid] = lst

print("=" * 75)
print(f"{'🧪 TEST_MODE' if TEST_MODE else '🚀 FULL'} | {len(video_shots)} video | caption {n_keep} shot | bỏ intro {n_intro} | bỏ ads {n_ads}")
print(f"⚡ Cấu hình: Batch Size = {BATCH_SIZE} (Cross-Video Streams)")
print(f"👉 Checkpoint JSONL: {OUTPUT_JSONL}")
print(f"📝 Realtime Log:     {LOG_FILE}")
print(f"📊 Dashboard Status: {STATUS_FILE}")
print("=" * 75)


🚀 FULL | 541 video | caption 48524 shot | bỏ intro 1494 | bỏ ads 0
⚡ Cấu hình: Batch Size = 7 (Cross-Video Streams)
👉 Checkpoint JSONL: /workspace/qwen3vl_8b_ctx_L24_L26.jsonl
📝 Realtime Log:     /workspace/caption_progress.log
📊 Dashboard Status: /workspace/caption_status.txt


--- 
## BƯỚC 3: Cài Đặt & Nâng Cấp Thư Viện Cần Thiết

In [12]:
# 1. Cài trực tiếp vào đúng Kernel/Venv đang mở bằng %pip
print("📦 Đang cài đặt thư viện cần thiết vào Kernel Python...")
%pip install -q "transformers>=4.57.0" "accelerate>=1.2.0" "qwen-vl-utils[decord]>=0.0.14" huggingface_hub

# 2. Bypass an toàn nms và nạp thư viện
import sys
import types
if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

import torch
import torchvision
import transformers
import qwen_vl_utils

print("\n" + "=" * 65)
print("✅ TẤT CẢ THƯ VIỆN ĐÃ ĐƯỢC NẠP THÀNH CÔNG VÀ AN TOÀN:")
print(f"  - PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - Torchvision: {torchvision.__version__} (Bypassed nms an toàn)")
print(f"  - Transformers: {transformers.__version__}")
print(f"  - Qwen-VL-Utils: {qwen_vl_utils.__file__}")
print("=" * 65)


📦 Đang cài đặt thư viện cần thiết vào Kernel Python...
Note: you may need to restart the kernel to use updated packages.

✅ TẤT CẢ THƯ VIỆN ĐÃ ĐƯỢC NẠP THÀNH CÔNG VÀ AN TOÀN:
  - PyTorch: 2.11.0+cu128 (CUDA: True)
  - Torchvision: 0.26.0+cu128 (Bypassed nms an toàn)
  - Transformers: 5.17.0
  - Qwen-VL-Utils: /venv/main/lib/python3.12/site-packages/qwen_vl_utils/__init__.py


---
## BƯỚC 4: Prompt V5 + Ngữ Cảnh Shot Trước

Thêm khối `[BỐI CẢNH: MÔ TẢ SHOT TRƯỚC]` (tối đa `PREV_CTX_MAX_CHARS` ký tự) và mục 5 hướng dẫn cách dùng: dùng để gọi thực thể nhất quán, KHÔNG chép lại.


In [13]:
# Cấu hình token đầu ra & Lấy mẫu video chuẩn (FPS=1.0, giữ 100% độ phân giải gốc cho OCR và thực thể)
MAX_NEW_TOKENS = 512
PREV_CTX_MAX_CHARS = 800      # cắt bớt caption shot trước để prompt không phình
VIDEO_FPS = 1.0
MIN_FRAMES = 6
MAX_FRAMES = 20

SYSTEM_PROMPT = (
    "Bạn là người hỗ trợ phân tích thị giác và lập chỉ mục dữ liệu video phục vụ hệ thống tìm kiếm (Video Retrieval). "
    "Mỗi đoạn video là MỘT CÚ MÁY ĐƠN LIÊN TỤC (single continuous shot, không có chuyển cảnh). "
    "Nhiệm vụ của bạn là quan sát diễn biến, bóc tách thực thể, màu sắc, số lượng và chi tiết thị giác "
    "một cách tự nhiên, chính xác và giàu thông tin nhất. "
    "Chỉ mô tả những gì NHÌN THẤY trong khung hình."
)

def build_user_prompt(asr_text: str, video_title: str = "", prev_caption: str = "") -> str:
    """Tạo prompt chuẩn hóa V5: Ràng buộc phương hướng không gian vật lý cho camera motion."""
    title_block = ""
    if video_title and video_title.strip():
        title_block = f'\n[TIÊU ĐỀ VIDEO / CHƯƠNG TRÌNH]:\n"{video_title.strip()}"\n'

    asr_context = (
        f'"{asr_text.strip()}"'
        if asr_text and asr_text.strip()
        else "(Không có lời thoại / Video chỉ có âm thanh hiện trường hoặc nhạc nền)"
    )

    prev_block = ""
    if prev_caption and prev_caption.strip():
        pc = prev_caption.strip()
        if len(pc) > PREV_CTX_MAX_CHARS:
            pc = pc[:PREV_CTX_MAX_CHARS].rsplit(" ", 1)[0] + " ..."
        prev_block = (
            "\n[BỐI CẢNH: MÔ TẢ SHOT TRƯỚC TRONG CÙNG VIDEO] (do máy sinh, có thể sai, KHÔNG phải shot đang xem):\n"
            f'"{pc}"\n'
        )

    return f"""Hãy phân tích cú máy video này phục vụ tìm kiếm thông tin:
{title_block}
[LỜI THOẠI ASR]:
{asr_context}
{prev_block}
HƯỚNG DẪN MÔ TẢ:

1. Trọng tâm chuyển động: Tập trung miêu tả hành động chính, người hoặc vật đang chuyển động và các sự vật bị tác động chuyển động theo. Nếu là cảnh tĩnh (không có chuyển động), tập trung miêu tả nội dung, bố cục và chi tiết các thực thể tĩnh.

2. Màu sắc & Số lượng (Kể cả đặc điểm hiển nhiên): Luôn chủ động gắn kèm màu sắc và số lượng (hoặc khoảng ước lượng nếu có) cho các thực thể/danh từ, ngay cả khi đó là thuộc tính hiển nhiên (ví dụ: củ dền màu đỏ/tím, cọng sả màu xanh/vàng, lòng đỏ trứng màu cam/vàng). Nếu chưa chắc chắn về tên vật thể hoặc màu sắc, dùng 'hoặc' (hoặc '/') để dự đoán gần đúng. Nếu chưa chắc chắn về số lượng, bắt buộc dùng khoảng (ví dụ: 2-3 cái, khoảng hơn 10 con)

3. Chữ trên màn hình: Đọc và ghi lại văn bản quan trọng NHÌN THẤY TRONG KHUNG HÌNH (dòng chữ chèn trên hình, phụ đề, biển hiệu, bảng điện tử, chữ trên logo, chữ trên trang phục hoặc đồ vật) NGUYÊN VĂN, đặt trong ngoặc kép. Chữ nhỏ hoặc mờ thì áp dụng đúng quy ước 'hoặc' ở mục 2: ghi phần đọc chắc, phần không chắc nêu vài khả năng gần đúng hoặc bỏ riêng phần đó. Điều phải tránh là viết ra MỘT dãy số/chuỗi hoàn chỉnh trông như đã đọc chắc chắn trong khi thực ra đang đoán. Với slide hoặc văn bản dài, chỉ ghi tiêu đề và các dòng mang thông tin chính, không chép lại toàn bộ.

4. Giới hạn nguồn thông tin: Caption chỉ được chứa thứ NHÌN THẤY trong khung hình. Tiêu đề video và lời thoại ASR được cấp cho bạn làm bối cảnh, nhưng KHÔNG phải là nguồn để viết caption:
   - Không nêu tên người, tên đội, tên chương trình, tên sự kiện, tên món ăn hay địa danh trừ khi chính chữ đó hiện trên màn hình và bạn đọc được.
   - Không trích lại lời thoại dưới bất kỳ hình thức nào, kể cả khi đang tả một người đang nói.
   - Không mô tả tiêu đề video như thể nó là chữ hiện trên màn hình. Đọc được bao nhiêu trên hình thì ghi bấy nhiêu; đọc không ra thì tả hình dạng ("logo tròn màu đỏ ở góc trên bên trái") rồi dừng lại, KHÔNG lấy tiêu đề video để điền vào.
   Lời nói thường không khớp với hình đang chiếu: phát thanh viên có thể đang nhắc tên một người không có trong khung hình. Không chắc ai là ai thì tả theo đặc điểm nhìn thấy (màu áo, số áo, vị trí), đừng gán tên.

5. Dùng bối cảnh shot trước: Mô tả shot trước chỉ giúp bạn hiểu mạch video (cùng nhân vật, cùng địa điểm, cùng chủ đề) để gọi thực thể nhất quán ("người đàn ông áo xanh" vẫn là người đó). Chỉ mô tả những gì NHÌN THẤY trong shot HIỆN TẠI: không chép lại câu chữ của shot trước, không nhắc vật thể hay hành động của shot trước nếu chúng không xuất hiện trong shot này. Nếu shot hiện tại khác hẳn shot trước thì bỏ qua bối cảnh đó.

XUẤT ĐẦU RA THEO ĐÚNG 2 MỤC SAU (KHÔNG THÊM LỜI DẪN):

- Chi tiết thị giác & Thực thể: (Miêu tả tự nhiên hành động, chuyển động của nhân vật, trang phục, màu sắc, số lượng nếu có, đạo cụ, đồ vật chính và chữ hiển thị trên màn hình nếu có)

- Góc máy & Không gian: (Cỡ cảnh: cận cảnh/trung cảnh/toàn cảnh. Sau đó BẮT BUỘC nêu chuyển động camera, chọn đúng một trong: tĩnh · lia ngang · lia dọc · zoom vào · zoom ra · bám theo chủ thể · bay trên cao. Camera đổi kiểu giữa chừng thì nêu cả hai theo thứ tự.
  ĐẶC BIỆT KHI CAMERA CÓ CHUYỂN ĐỘNG (lia ngang, lia dọc, zoom, bám theo): BẮT BUỘC nêu rõ PHƯƠNG HƯỚNG KHÔNG GIAN VẬT LÝ (từ trái sang phải / từ phải sang trái / từ trên xuống dưới / từ dưới lên trên / từ trước ra sau / từ xa lại gần / tiến lại gần / lùi ra xa) kết hợp với mốc đồ vật hoặc nhân vật nếu có. Cú pháp chuẩn: [chuyển động] [phương hướng vật lý] (từ [mốc vật thể A] sang [mốc vật thể B]). Ví dụ: 'camera lia ngang từ trái sang phải (từ bó hoa sang đĩa tôm và thịt)', 'camera lia dọc từ trên xuống dưới (từ biển hiệu trên cao xuống bậc thềm vỉa hè)'.
  Cách xác định camera tĩnh: chỉ chọn "tĩnh" khi khung nền KHÔNG đổi trong suốt đoạn. Nếu hậu cảnh trôi qua khung hình, hoặc vật thể đứng yên dịch chuyển trong khung, thì camera đang chuyển động — không phải tĩnh.
  Cuối cùng nêu không gian trong nhà/ngoài trời.)

VÍ DỤ MẪU:

Ví dụ 1:
- Chi tiết thị giác & Thực thể: Hai công nhân vệ sinh mặc áo bảo hộ màu cam và quần xanh đậm đang dùng chổi tre quét lá khô màu nâu trên vỉa hè lát gạch xám. Một người đẩy chiếc xe rác màu xanh lá có 2 bánh, bên trong chứa khoảng 3-4 bao rác màu đen. Phía dưới khung hình có dòng chữ "THU GOM RÁC SINH HOẠT".
- Góc máy & Không gian: Trung cảnh, camera bám theo chủ thể khi hai người di chuyển dọc vỉa hè; không gian ngoài trời trên đường phố có hàng cây hai bên.

Ví dụ 2:
- Chi tiết thị giác & Thực thể: Một chiếc xe buýt màu vàng và trắng đang rẽ trái vào bến, trên kính chắn gió có bảng số tuyến. Khoảng 5-7 hành khách đứng chờ dưới mái che màu xanh dương, trong đó một phụ nữ mặc áo sơ mi trắng đang cầm điện thoại. Biển hiệu tại bến ghi "TRAM DUNG XE BUYT SO 12".
- Góc máy & Không gian: Toàn cảnh, camera lia ngang từ trái sang phải theo hướng xe chạy rồi dừng lại; không gian ngoài trời.
"""

print(f"✅ Đã cấu hình Prompt V4 (Khóa triệt để ASR/Title, Giới hạn slide dài, Camera danh sách đóng)!")

✅ Đã cấu hình Prompt V4 (Khóa triệt để ASR/Title, Giới hạn slide dài, Camera danh sách đóng)!


--- 
## BƯỚC 5: Nạp Model Qwen3-VL-8B-Instruct Với device_map='auto'

Model **`Qwen/Qwen3-VL-8B-Instruct`** (~16.5 GB weights). Khi chạy trên Kaggle GPU T4 x2 (Dual T4 - 32GB VRAM), mô hình sẽ tự động chia đều lên 2 GPU, nạp 100% trong VRAM không cần offload CPU!

In [14]:
import time
import sys
import types
import torch

if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

# 🚀 ÉP XUNG TOÁN HỌC TENSOR CORES (TF32) CHO 2x RTX 5060 Ti
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MIN_PIXELS = 128 * 28 * 28   # 100,352 pixels
MAX_PIXELS = 360 * 640       # 230,400 pixels

if 'model' in globals() and model is not None and 'processor' in globals() and processor is not None:
    print("⚡ [BỎ QUA NẠP LẠI] Model và Processor đã có sẵn trong VRAM!")
    print("   => Đại ca có thể thoải mái bấm 'Run all below' từ Bước 4 mà không sợ tốn thời gian nạp lại.")
    # Đảm bảo padding_side = "left" cho batch generation
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"
else:
    from transformers import AutoProcessor
    print(f"[1/2] Đang nạp Processor cho model: {MODEL_ID}...")
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS
    )
    # Bắt buộc padding_side="left" cho decoder-only text generation khi batching
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

    print(f"[2/2] Đang nạp Model weights với dtype=bfloat16, device_map='auto' + SDPA Native FlashAttention...")
    start_load = time.time()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    import transformers
    print(f"  transformers đang nạp trong kernel: {transformers.__version__}")
    if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (4, 57):
        raise RuntimeError(
            f"transformers {transformers.__version__} chưa có Qwen3-VL (cần >= 4.57). "
            "Bản mới đã được pip cài ở Bước 3 nhưng kernel vẫn giữ bản cũ: Restart Kernel rồi chạy lại từ đầu.")
    from transformers import Qwen3VLForConditionalGeneration

    num_gpus = torch.cuda.device_count()
    max_memory = {i: "10GiB" for i in range(num_gpus)} if num_gpus > 1 else None
    
    # Nạp model với attn_implementation="sdpa" để kích hoạt nhân FlashAttention tối ưu cho Tensor Cores
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, 
        torch_dtype=dtype, 
        device_map="auto", 
        max_memory=max_memory,
        attn_implementation="sdpa"
    )
    print(f"  => ✅ Nạp xong model Qwen3-VL-8B trong {round(time.time() - start_load, 2)}s")

    if model is None:
        raise RuntimeError(f"Không thể nạp model {MODEL_ID}!")

    model.eval()

print("\n--- THỐNG KÊ PHÂN BỔ VRAM SAU KHI NẠP ---")
for i in range(torch.cuda.device_count()):
    alloc_gb = round(torch.cuda.memory_allocated(i) / (1024**3), 2)
    res_gb = round(torch.cuda.memory_reserved(i) / (1024**3), 2)
    print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): Allocated = {alloc_gb} GB | Reserved = {res_gb} GB")
if hasattr(model, "hf_device_map"): 
    print(f"  [Device Map]: Các layers được phân bổ trên thiết bị: {set(model.hf_device_map.values())}")
print("-------------------------------------------")


⚡ [BỎ QUA NẠP LẠI] Model và Processor đã có sẵn trong VRAM!
   => Đại ca có thể thoải mái bấm 'Run all below' từ Bước 4 mà không sợ tốn thời gian nạp lại.

--- THỐNG KÊ PHÂN BỔ VRAM SAU KHI NẠP ---
  GPU 0 (NVIDIA GeForce RTX 5090): Allocated = 16.34 GB | Reserved = 30.65 GB
-------------------------------------------


---
## BƯỚC 6: Hàm Suy Luận


In [15]:
import time
import sys
import types
import torch
import json
import os

if 'torchvision._meta_registrations' not in sys.modules:
    sys.modules['torchvision._meta_registrations'] = types.ModuleType('torchvision._meta_registrations')

from qwen_vl_utils import process_vision_info


def get_video_title(video_id: str, media_info_dir: str) -> str:
    info_path = os.path.join(media_info_dir, f"{video_id}.json")
    if not os.path.exists(info_path):
        return ""
    try:
        with open(info_path, "r", encoding="utf-8") as f:
            return json.load(f).get("title", "").strip()
    except Exception:
        return ""


def get_asr_context(video_id: str, shot_start: float, shot_end: float, asr_dir: str):
    """Ghép các segment ASR giao với khoảng thời gian của shot."""
    asr_path = os.path.join(asr_dir, f"{video_id}.json")
    if not os.path.exists(asr_path):
        return "", 0
    try:
        with open(asr_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        segments = data if isinstance(data, list) else data.get("segments", [])
        texts = [seg.get("text", "").strip() for seg in segments
                 if seg.get("text", "").strip()
                 and max(seg.get("start", 0), shot_start) < min(seg.get("end", 0), shot_end)]
        return " ".join(texts), len(texts)
    except Exception:
        return "", 0


def caption_batch_shots(model, processor, batch_items: list, fps: float = VIDEO_FPS,
                        min_frames: int = MIN_FRAMES, max_frames: int = MAX_FRAMES,
                        max_new_tokens: int = MAX_NEW_TOKENS, repetition_penalty: float = 1.05):
    """
    Suy luận song song cả batch (Batch Size = 1 hoặc 2) trên GPU 5060 Ti:
    - items là danh sách dict: [{'video_path', 'asr_text', 'video_title', 'prev_caption'}, ...]
    - Tận dụng GEMM Matrix Multiplication giúp Tensor Cores chạy hết công suất!
    """
    if not batch_items:
        return []
    start_time = time.time()

    # Đảm bảo padding_side = "left" cho decoder-only LLM
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

    messages_list = []
    for it in batch_items:
        user_prompt = build_user_prompt(
            asr_text=it.get("asr_text", ""),
            video_title=it.get("video_title", ""),
            prev_caption=it.get("prev_caption", "")
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "video", "video": it["video_path"], "fps": fps, "min_frames": min_frames, "max_frames": max_frames},
                {"type": "text", "text": user_prompt},
            ]},
        ]
        messages_list.append(messages)

    prompt_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_list
    ]

    # Xử lý vision info cho cả batch (qwen_vl_utils hỗ trợ trực tiếp list of conversations)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages_list, image_patch_size=16, return_video_kwargs=True, return_video_metadata=True
    )
    video_metadatas = None
    if video_inputs is not None:
        video_inputs, video_metadatas = map(list, zip(*video_inputs))

    inputs = processor(
        text=prompt_texts,
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        do_resize=False,
        padding=True,
        return_tensors="pt",
        **video_kwargs
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=repetition_penalty
        )

    # Vì padding_side='left', generated tokens nằm sau độ dài input_ids lớn nhất
    input_len = inputs["input_ids"].shape[1]
    gen_ids = generated_ids[:, input_len:]
    captions = processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    batch_duration = round(time.time() - start_time, 2)
    per_item_time = round(batch_duration / max(1, len(batch_items)), 2)

    results = []
    for i in range(len(batch_items)):
        gen_tokens_count = int((gen_ids[i] != processor.tokenizer.pad_token_id).sum().item()) if hasattr(processor.tokenizer, "pad_token_id") and processor.tokenizer.pad_token_id is not None else int(gen_ids.shape[1])
        results.append({
            "caption": captions[i].strip(),
            "infer_time_sec": per_item_time,
            "batch_duration_sec": batch_duration,
            "input_tokens": int(input_len),
            "generated_tokens": gen_tokens_count
        })

    return results

print("✅ Đã khởi tạo hàm suy luận Batch (hỗ trợ Cross-Video Multi-Stream Batching)!")


✅ Đã khởi tạo hàm suy luận Batch (hỗ trợ Cross-Video Multi-Stream Batching)!


---
## BƯỚC 7: Chạy Tuần Tự Theo Video (Nối Ngữ Cảnh)

Shot trong một video phải chạy theo thứ tự, vì shot sau cần caption của shot trước. Khi chạy lại, notebook resume theo `shot_file` và nạp lại caption cũ làm ngữ cảnh.


In [ ]:
import gc
import time
import json
import os
import logging
from IPython.display import display, HTML

# Ẩn thông báo lặp vô hại
_NOISY_KEYS = ("using torchcodec to read video", "cap_pixels_per_frame")
class _DropNoisy(logging.Filter):
    def filter(self, record):
        return not any(k in record.getMessage() for k in _NOISY_KEYS)
for _lg in [logging.getLogger()] + [logging.getLogger(n) for n in list(logging.root.manager.loggerDict)]:
    _lg.addFilter(_DropNoisy())
    for _h in _lg.handlers:
        _h.addFilter(_DropNoisy())

# 🎯 CẤU HÌNH KIỂM SOÁT
FORCE_RERUN = False                      # False: Kế thừa 100% checkpoint cũ, không bao giờ chạy lại!

# 1. ĐỌC VÀ KHÔI PHỤC CHECKPOINT ĐÃ CÓ
done = {}                                # shot_file -> caption (để resume và khôi phục ngữ cảnh)
if os.path.exists(OUTPUT_JSONL) and not FORCE_RERUN:
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                done[r["shot_file"]] = r["caption"]
            except Exception:
                pass

# Tính tổng số shot cần caption trong phạm vi
total_shots_in_scope = sum(1 for lst in video_shots.values() for _, r in lst if r is None)
already_done_count = len([k for k in done.keys() if any(k == os.path.basename(p) for lst in video_shots.values() for p, r in lst if r is None)])
shots_to_run_count = total_shots_in_scope - already_done_count

print("=" * 80)
print(f"🚀 BẮT ĐẦU CHẠY: {total_shots_in_scope} shot tổng cộng")
print(f"  • Đã có sẵn trong Checkpoint: {already_done_count} shot (BỎ QUA KHÔNG CHẠY LẠI)")
print(f"  • Cần xử lý tiếp:             {shots_to_run_count} shot")
print(f"  • Chế độ:                     Batch Size = {BATCH_SIZE} (Cross-Video)")
print(f"  • File lưu kết quả:           {OUTPUT_JSONL}")
print(f"  • File log realtime:          {LOG_FILE} (theo dõi bằng lệnh: tail -f {LOG_FILE})")
print(f"  • File trạng thái:            {STATUS_FILE}")
print("=" * 80)

# Hàm ghi log realtime và update file status
def log_progress(msg: str):
    timestamp_str = time.strftime('%Y-%m-%d %H:%M:%S')
    log_line = f"[{timestamp_str}] {msg}"
    print(log_line)
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as lf:
            lf.write(log_line + "\n")
            lf.flush()
            os.fsync(lf.fileno())
    except Exception:
        pass

def update_status_file(finished_cnt, total_cnt, current_shot, current_prev, last_infer_sec, eta_min, vram_info=""):
    try:
        pct = (finished_cnt / max(1, total_cnt)) * 100
        status_text = (
            "======================================================================\n"
            "📊 BẢNG THEO DÕI TIẾN ĐỘ TRỰC TIẾP - QWEN3-VL-8B CAPTIONING\n"
            "======================================================================\n"
            f"⏱️  Cập nhật lúc:   {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"🚀  Tiến độ:        {finished_cnt}/{total_cnt} shots ({pct:.1f}%)\n"
            f"⚡  Tốc độ:         ~{last_infer_sec:.2f}s / shot (Batch Size = {BATCH_SIZE})\n"
            f"⏳  ETA còn lại:    ~{eta_min:.1f} phút\n"
            f"🎯  Shot vừa xong:  {current_shot}\n"
            f"🔗  Ngữ cảnh nối:   {current_prev or '(bắt đầu video)'}\n"
            f"📂  Checkpoint:     {OUTPUT_JSONL}\n"
            f"📝  Log Realtime:   {LOG_FILE}\n"
            f"🖥️  {vram_info}\n"
            "======================================================================\n"
        )
        with open(STATUS_FILE, "w", encoding="utf-8") as sf:
            sf.write(status_text)
            sf.flush()
    except Exception:
        pass

# ==============================================================================
# 2. XÂY DỰNG DANH SÁCH VIDEO QUEUE & KHÔI PHỤC NGỮ CẢNH TỪNG VIDEO
# ==============================================================================
# Đối với mỗi video:
# - Quét qua các shot theo thứ tự thời gian.
# - Shot nào đã có trong done -> cập nhật prev_caption của video đó!
# - Shot nào chưa có trong done -> đưa vào remaining_shots cần chạy!
video_queues = []
for vid, lst in video_shots.items():
    title = get_video_title(vid, MEDIA_INFO_DIR)
    prev_caption = ""
    prev_file = ""
    remaining = []
    
    for path, reason in lst:
        fname = os.path.basename(path)
        if reason is not None:
            # Shot bị bỏ qua (intro hoặc ads): KHÔNG đổi prev_caption, giữ nguyên ngữ cảnh của shot hợp lệ trước
            continue
        
        if fname in done:
            # Shot ĐÃ HOÀN THÀNH trong checkpoint: Khôi phục ngữ cảnh của nó!
            prev_caption = done[fname]
            prev_file = fname
            continue
            
        # Shot CHƯA HOÀN THÀNH:
        _, shot_idx, s, e = parse_shot_filename(fname)
        asr_text, seg_count = get_asr_context(vid, s, e, ASR_DIR)
        remaining.append({
            "vid": vid,
            "path": path,
            "fname": fname,
            "shot_idx": shot_idx,
            "start_sec": s,
            "end_sec": e,
            "duration": e - s,
            "title": title,
            "asr_text": asr_text,
            "seg_count": seg_count,
        })
        
    if remaining:
        video_queues.append({
            "vid": vid,
            "title": title,
            "prev_caption": prev_caption,  # Ngữ cảnh kế thừa chuẩn xác từ checkpoint!
            "prev_file": prev_file,
            "remaining_shots": remaining
        })

log_progress(f"Khởi động bộ điều phối: {len(video_queues)} video có shot cần chạy tiếp.")

# ==============================================================================
# 3. VÒNG LẶP ĐIỀU PHỐI CROSS-VIDEO BATCHING (BATCH SIZE = 2)
# ==============================================================================
active_streams = []               # Danh sách các stream video đang active (tối đa BATCH_SIZE)
n_run = 0
t_session = time.time()
errors = []

# Mở sẵn file JSONL ở chế độ append để ghi liên tục
jsonl_file = open(OUTPUT_JSONL, "a", encoding="utf-8")

try:
    while active_streams or video_queues:
        # 1. Bổ sung stream nếu còn chỗ trống
        while len(active_streams) < BATCH_SIZE and video_queues:
            next_v = video_queues.pop(0)
            active_streams.append(next_v)

        if not active_streams:
            break

        # 2. Tạo batch gồm tối đa BATCH_SIZE shot từ các video khác nhau
        batch_tasks = []
        for stream in active_streams:
            if stream["remaining_shots"]:
                shot_data = stream["remaining_shots"].pop(0)
                batch_tasks.append({
                    "stream": stream,
                    "shot_data": shot_data,
                    "video_path": shot_data["path"],
                    "asr_text": shot_data["asr_text"],
                    "video_title": shot_data["title"],
                    "prev_caption": stream["prev_caption"]  # TRUYỀN NGỮ CẢNH SHOT TRƯỚC CỦA CHÍNH VIDEO NÀY!
                })

        if not batch_tasks:
            break

        # 3. Gửi batch vào GPU suy luận
        try:
            results = caption_batch_shots(
                model=model,
                processor=processor,
                batch_items=[{
                    "video_path": bt["video_path"],
                    "asr_text": bt["asr_text"],
                    "video_title": bt["video_title"],
                    "prev_caption": bt["prev_caption"]
                } for bt in batch_tasks],
                max_new_tokens=MAX_NEW_TOKENS
            )
        except torch.cuda.OutOfMemoryError as ex:
            gc.collect(); torch.cuda.empty_cache()
            log_progress(f"❌ CẢNH BÁO OOM BATCH: {ex} -> Thu hồi bộ nhớ và bỏ qua batch.")
            errors.append(("BATCH_OOM", str(ex)))
            continue
        except Exception as ex:
            log_progress(f"❌ LỖI BATCH: {ex}")
            errors.append(("BATCH_ERR", str(ex)))
            continue

        # 4. Xử lý kết quả từng shot trong batch
        last_item_time = 0.0
        last_shot_name = ""
        last_prev_name = ""
        for bt, res in zip(batch_tasks, results):
            stream = bt["stream"]
            sd = bt["shot_data"]
            n_run += 1
            last_item_time = res["infer_time_sec"]
            last_shot_name = sd["fname"]
            last_prev_name = stream["prev_file"]

            rec = {
                "video_id": sd["vid"],
                "shot_file": sd["fname"],
                "shot_idx": sd["shot_idx"],
                "shot_start_sec": sd["start_sec"],
                "shot_end_sec": sd["end_sec"],
                "duration_sec": sd["duration"],
                "video_title": sd["title"],
                "asr_segment_count": sd["seg_count"],
                "asr_context": sd["asr_text"],
                "prev_shot_file": stream["prev_file"],
                "prev_caption": stream["prev_caption"],
                "caption": res["caption"],
                "model_id": MODEL_ID,
                "infer_time_sec": res["infer_time_sec"],
                "batch_duration_sec": res.get("batch_duration_sec", 0.0),
                "input_tokens": res["input_tokens"],
                "generated_tokens": res["generated_tokens"],
                "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
            }

            # Ghi checkpoint ngay lập tức
            jsonl_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
            jsonl_file.flush()

            # CẬP NHẬT NGỮ CẢNH CHO SHOT TIẾP THEO CỦA CHÍNH VIDEO NÀY:
            stream["prev_caption"] = res["caption"]
            stream["prev_file"] = sd["fname"]
            done[sd["fname"]] = res["caption"]

            # Tính toán tiến độ & ETA
            curr_done = len(done)
            el_time = time.time() - t_session
            eta_min = (el_time / max(1, n_run)) * (total_shots_in_scope - curr_done) / 60.0

            log_msg = (
                f"[{curr_done:04d}/{total_shots_in_scope}] {sd['fname']} | "
                f"prev={rec['prev_shot_file'] or '-'} | "
                f"{res['infer_time_sec']}s (batch: {res.get('batch_duration_sec', 0.0)}s) | "
                f"in={res['input_tokens']} out={res['generated_tokens']} tok | "
                f"ETA ~{eta_min:.1f}m"
            )
            log_progress(log_msg)

        # 5. Cập nhật file status tóm tắt
        vram_str = " | ".join([f"GPU {i}: {round(torch.cuda.memory_allocated(i)/(1024**3), 1)}GB" for i in range(torch.cuda.device_count())]) if torch.cuda.is_available() else ""
        curr_done = len(done)
        el_time = time.time() - t_session
        eta_min = (el_time / max(1, n_run)) * (total_shots_in_scope - curr_done) / 60.0
        update_status_file(curr_done, total_shots_in_scope, last_shot_name, last_prev_name, last_item_time, eta_min, vram_str)

        # 6. Loại bỏ những stream đã hết shot để vòng lặp sau nạp video mới
        active_streams = [s for s in active_streams if s["remaining_shots"]]

finally:
    jsonl_file.close()

log_progress(f"🏁 KẾT THÚC PHIÊN: Hoàn thành thêm {n_run} shot trong {time.time() - t_session:.1f}s | Lỗi: {len(errors)}")
print("=" * 80)


---
## BƯỚC 8: Xuất JSON & Tóm Tắt


In [ ]:
# Xuất JSON gọn và tóm tắt
OUTPUT_JSON = OUTPUT_JSONL[:-1]          # .jsonl -> .json
records = []
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        try: records.append(json.loads(line))
        except Exception: pass
records.sort(key=lambda r: (r["video_id"], r["shot_idx"]))
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"💾 {len(records)} shot -> {OUTPUT_JSON}")
if records:
    n = len(records)
    print(f"  • Suy luận TB: {sum(r['infer_time_sec'] for r in records)/n:.2f}s/shot | out TB: {sum(r['generated_tokens'] for r in records)/n:.0f} tok")
    ctx = [r for r in records if "input_tokens_no_ctx" in r]
    if ctx:
        print(f"  • Token đầu vào tăng thêm do ngữ cảnh: +{sum(r['input_tokens'] - r['input_tokens_no_ctx'] for r in ctx)/len(ctx):.0f} tok/shot")
